In [1]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.preprocessing import LabelEncoder
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU"))

gpus = tf.config.list_physical_devices("GPU")
if gpus:
    try:
        tf.config.set_visible_devices(gpus[0], "GPU")
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except Exception as e:
        print("GPU setup warning:", e)

# 1. Wczytanie danych
csv_path = r"C:\Users\tomas\Desktop\github\upp_csde\metody_si\projekt\dane\merged_data.csv"
df = pd.read_csv(csv_path, encoding="utf-8")

if df.empty:
    raise ValueError(f"DataFrame loaded from {csv_path} is empty. Check the file path and contents.")

if "POST" not in df.columns:
    raise ValueError("Brak kolumny POST w danych.")

# 2. Data
df["DATA"] = pd.to_datetime(dict(year=df["ROK"], month=df["MC"], day=df["DZ"]))

# 3. Sortowanie po lokalizacji i czasie
df = df.sort_values(["POST", "DATA"]).reset_index(drop=True)

# 4. Target = średnia temperatura z następnego dnia, liczona osobno per lokalizacja
df["TARGET"] = df.groupby("POST")["STD"].shift(-1)

# 5. Lagi liczone osobno dla każdej lokalizacji
df["STD_lag1"] = df.groupby("POST")["STD"].shift(1)
df["TMAX_lag1"] = df.groupby("POST")["TMAX"].shift(1)
df["TMIN_lag1"] = df.groupby("POST")["TMIN"].shift(1)

# opcjonalnie możesz od razu dodać więcej lagów:
# df["STD_lag2"] = df.groupby("POST")["STD"].shift(2)
# df["TMAX_lag2"] = df.groupby("POST")["TMAX"].shift(2)
# df["TMIN_lag2"] = df.groupby("POST")["TMIN"].shift(2)

# 6. Usunięcie braków
need_cols = ["STD_lag1", "TMAX_lag1", "TMIN_lag1", "TARGET", "POST"]
df = df.dropna(subset=need_cols).copy()

if df.empty:
    raise ValueError("No data left after creating lags / dropping NA. Check input data.")

# 7. Kodowanie lokalizacji jako liczby całkowite do embeddingu
le = LabelEncoder()
df["POST_ID"] = le.fit_transform(df["POST"])

num_locations = df["POST_ID"].nunique()
embedding_dim = min(16, max(4, num_locations // 2))

print("Liczba lokalizacji:", num_locations)
print("Embedding dim:", embedding_dim)

# 8. Cechy numeryczne
num_features = ["STD_lag1", "TMAX_lag1", "TMIN_lag1"]

X_num = df[num_features].apply(pd.to_numeric, errors="coerce").values.astype(np.float32)
X_loc = df["POST_ID"].values.astype(np.int32)
y = pd.to_numeric(df["TARGET"], errors="coerce").values.astype(np.float32)

if np.isnan(X_num).any() or np.isnan(y).any():
    raise ValueError("W danych są NaN po konwersji do liczb.")

# 9. Podział czasowy
split = int(len(df) * 0.8)
if split < 1:
    split = 1 if len(df) > 1 else len(df)

X_num_train, X_num_test = X_num[:split], X_num[split:]
X_loc_train, X_loc_test = X_loc[:split], X_loc[split:]
y_train, y_test = y[:split], y[split:]

if X_num_train.shape[0] == 0:
    raise ValueError("Training set is empty after split. Increase data or adjust split ratio.")

if X_num_test.shape[0] == 0:
    if X_num_train.shape[0] > 1:
        X_num_test = X_num_train[-1:].copy()
        X_loc_test = X_loc_train[-1:].copy()
        y_test = y_train[-1:].copy()

        X_num_train = X_num_train[:-1]
        X_loc_train = X_loc_train[:-1]
        y_train = y_train[:-1]
    else:
        raise ValueError("Not enough data for testing. Need at least 2 samples.")

# 10. Skalowanie cech numerycznych
mean = X_num_train.mean(axis=0)
std = X_num_train.std(axis=0)
std_fixed = np.where(std == 0, 1.0, std)

X_num_train = ((X_num_train - mean) / std_fixed).astype(np.float32)
X_num_test = ((X_num_test - mean) / std_fixed).astype(np.float32)

# 11. Model z dwoma wejściami: numerycznym i embeddingiem lokalizacji
num_input = keras.Input(shape=(X_num_train.shape[1],), name="num_input")
loc_input = keras.Input(shape=(1,), dtype="int32", name="loc_input")

loc_embedding = layers.Embedding(
    input_dim=num_locations,
    output_dim=embedding_dim,
    name="location_embedding"
)(loc_input)

loc_vec = layers.Flatten()(loc_embedding)

x = layers.Concatenate()([num_input, loc_vec])
x = layers.Dense(32, activation="relu")(x)
x = layers.Dense(16, activation="relu")(x)
x = layers.Dense(8, activation="relu")(x)
output = layers.Dense(1)(x)

model = keras.Model(inputs=[num_input, loc_input], outputs=output)

# 12. Kompilacja
model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

# 13. Early stopping
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=30,
    restore_best_weights=True
)

# 14. Trenowanie
fit_kwargs = dict(
    x={"num_input": X_num_train, "loc_input": X_loc_train},
    y=y_train,
    epochs=200,
    batch_size=256,
    callbacks=[early_stop],
    verbose=1
)

if X_num_train.shape[0] >= 10:
    fit_kwargs["validation_split"] = 0.2
else:
    fit_kwargs["validation_data"] = (
        {"num_input": X_num_test, "loc_input": X_loc_test},
        y_test
    )

history = model.fit(**fit_kwargs)

# 15. Predykcja
y_pred = model.predict(
    {"num_input": X_num_test, "loc_input": X_loc_test},
    verbose=0
).flatten()

# 16. Ocena
mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5

print("MAE:", round(mae, 3))
print("RMSE:", round(rmse, 3))

# 17. Wyniki
wyniki = pd.DataFrame({
    "data": df["DATA"].iloc[split:split + len(y_test)].values,
    "POST": df["POST"].iloc[split:split + len(y_test)].values,
    "rzeczywiste": y_test,
    "przewidywane": y_pred
})

print(wyniki.head(20))

# 18. Zapis modelu
model.save("temperature_model_embedding.keras")

# 19. Opcjonalnie: podejrzenie nauczonych embeddingów lokalizacji
embedding_weights = model.get_layer("location_embedding").get_weights()[0]
embedding_df = pd.DataFrame(
    embedding_weights,
    index=le.classes_
)

print("\nPrzykładowe embeddingi lokalizacji:")
print(embedding_df.head(10))

TensorFlow: 2.10.1
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Liczba lokalizacji: 227
Embedding dim: 16
Epoch 1/200
2656/2656 [==============================] - 28s 3ms/step - loss: 14.9661 - mae: 2.8612 - val_loss: 26.7567 - val_mae: 4.3701
Epoch 2/200
2656/2656 [==============================] - 8s 3ms/step - loss: 10.9060 - mae: 2.5630 - val_loss: 21.4945 - val_mae: 3.8211
Epoch 3/200
2656/2656 [==============================] - 8s 3ms/step - loss: 10.8329 - mae: 2.5547 - val_loss: 18.1035 - val_mae: 3.4581
Epoch 4/200
2656/2656 [==============================] - 8s 3ms/step - loss: 10.8053 - mae: 2.5511 - val_loss: 16.2531 - val_mae: 3.2610
Epoch 5/200
2656/2656 [==============================] - 8s 3ms/step - loss: 10.7843 - mae: 2.5484 - val_loss: 15.1230 - val_mae: 3.1257
Epoch 6/200
2656/2656 [==============================] - 8s 3ms/step - loss: 10.7723 - mae: 2.5471 - val_loss: 14.6408 - val_mae: 3.0574
Epoch 7/200
2656/2656 [=====================

In [25]:
df_raw = pd.read_csv(csv_path, encoding="utf-8")
df_raw["DATA"] = pd.to_datetime(dict(year=df_raw["ROK"], month=df_raw["MC"], day=df_raw["DZ"]))
df_raw = df_raw.sort_values(["POST", "DATA"]).reset_index(drop=True)

# --------------------------------------------------
# FUNKCJA 1: pojedyncza predykcja
# --------------------------------------------------
def predict_temperature(post_name, std_lag1, tmax_lag1, tmin_lag1):
    if post_name not in le.classes_:
        raise ValueError(f"Lokalizacja '{post_name}' nie istnieje w danych.")

    post_id = le.transform([post_name])[0]

    X_num_manual = np.array([[std_lag1, tmax_lag1, tmin_lag1]], dtype=np.float32)
    X_loc_manual = np.array([[post_id]], dtype=np.int32)

    X_num_manual = ((X_num_manual - mean) / std_fixed).astype(np.float32)

    prediction = model.predict(
        {"num_input": X_num_manual, "loc_input": X_loc_manual},
        verbose=0
    ).flatten()[0]

    return float(prediction)

def get_last_known_row(post_name):
    loc_df = df_raw[df_raw["POST"] == post_name].sort_values("DATA")

    if loc_df.empty:
        raise ValueError(f"Brak danych dla lokalizacji: {post_name}")

    last_row = loc_df.iloc[-1]

    return {
        "date": pd.to_datetime(last_row["DATA"]),
        "std": float(last_row["STD"]),
        "tmax": float(last_row["TMAX"]),
        "tmin": float(last_row["TMIN"])
    }

In [26]:
# --------------------------------------------------
# FUNKCJA 2: pobranie ostatnich danych lokalizacji
# --------------------------------------------------
def get_last_values_for_location(post_name):
    loc_df = df[df["POST"] == post_name].sort_values("DATA")

    if loc_df.empty:
        raise ValueError(f"Brak danych dla lokalizacji: {post_name}")

    last_row = loc_df.iloc[-1]

    return {
        "last_date": last_row["DATA"],
        "std_lag1": float(last_row["STD"]),
        "tmax_lag1": float(last_row["TMAX"]),
        "tmin_lag1": float(last_row["TMIN"])
    }

In [27]:
# --------------------------------------------------
# FUNKCJA 3: prognoza wielodniowa dzien po dniu
# --------------------------------------------------
def forecast_recursive_until_date(post_name, end_date, tmax_delta=2.0, tmin_delta=2.0):
    last_row = get_last_known_row(post_name)

    last_known_date = pd.to_datetime(last_row["date"])
    end_date = pd.to_datetime(end_date)

    if end_date <= last_known_date:
        raise ValueError(
            f"Data koncowa musi byc pozniejsza niz ostatnia data w CSV dla {post_name} "
            f"({last_known_date.date()})."
        )

    current_date = last_known_date + pd.Timedelta(days=1)

    std_lag1 = last_row["std"]
    tmax_lag1 = last_row["tmax"]
    tmin_lag1 = last_row["tmin"]

    results = []

    print(f"\nOstatnia data w raw CSV: {last_known_date.date()}")
    print(f"Start prognozy: {current_date.date()}")
    print(f"Koniec prognozy: {end_date.date()}")

    while current_date <= end_date:
        pred_std = predict_temperature(post_name, std_lag1, tmax_lag1, tmin_lag1)

        results.append({
            "data": current_date,
            "POST": post_name,
            "STD_lag1_used": std_lag1,
            "TMAX_lag1_used": tmax_lag1,
            "TMIN_lag1_used": tmin_lag1,
            "predicted_STD": pred_std
        })

        std_lag1 = pred_std
        tmax_lag1 = pred_std + tmax_delta
        tmin_lag1 = pred_std - tmin_delta

        print(
            f"{current_date.date()} | "
            f"wejscie: STD={results[-1]['STD_lag1_used']:.2f}, "
            f"TMAX={results[-1]['TMAX_lag1_used']:.2f}, "
            f"TMIN={results[-1]['TMIN_lag1_used']:.2f} "
            f"-> prognoza STD={pred_std:.2f}"
        )

        current_date += pd.Timedelta(days=1)

    return pd.DataFrame(results)

In [29]:
# --------------------------------------------------
# PRZYKLAD UZYCIA
# --------------------------------------------------

prognoza = forecast_recursive_until_date(
    post_name="GNIEZNO",
    end_date="2026-06-07"
)

print("\nWynik koncowy:")
print(prognoza)


Ostatnia data w raw CSV: 2014-12-31
Start prognozy: 2015-01-01
Koniec prognozy: 2026-06-07
2015-01-01 | wejscie: STD=-0.40, TMAX=3.60, TMIN=-4.60 -> prognoza STD=2.12
2015-01-02 | wejscie: STD=2.12, TMAX=4.12, TMIN=0.12 -> prognoza STD=2.30
2015-01-03 | wejscie: STD=2.30, TMAX=4.30, TMIN=0.30 -> prognoza STD=2.42
2015-01-04 | wejscie: STD=2.42, TMAX=4.42, TMIN=0.42 -> prognoza STD=2.51
2015-01-05 | wejscie: STD=2.51, TMAX=4.51, TMIN=0.51 -> prognoza STD=2.57
2015-01-06 | wejscie: STD=2.57, TMAX=4.57, TMIN=0.57 -> prognoza STD=2.62
2015-01-07 | wejscie: STD=2.62, TMAX=4.62, TMIN=0.62 -> prognoza STD=2.66
2015-01-08 | wejscie: STD=2.66, TMAX=4.66, TMIN=0.66 -> prognoza STD=2.68
2015-01-09 | wejscie: STD=2.68, TMAX=4.68, TMIN=0.68 -> prognoza STD=2.70
2015-01-10 | wejscie: STD=2.70, TMAX=4.70, TMIN=0.70 -> prognoza STD=2.71
2015-01-11 | wejscie: STD=2.71, TMAX=4.71, TMIN=0.71 -> prognoza STD=2.72
2015-01-12 | wejscie: STD=2.72, TMAX=4.72, TMIN=0.72 -> prognoza STD=2.73
2015-01-13 | wejsc